## Agenda

#### 1 Create spark session

  - Create spark session
  - installing some libraries
  - Restarting python (in case previous libraries installation asks to do so)
  - Importing some libraries

#### 2 create dataframe

-     1 By reading a file in the Volume(a csv, json, etc.) FOOL(Format, Option, Option, Load)
    -  using  spark.read.format().option().option().load()
- 2 By reading a table where the data sits using spark.read.table
- 3 Check the notebook "3_Customer_Dataframe_Creation".  using spark.createDataFrame(data=data_list, schema=data_schema)

###### 2.1 Eyeball to spot possible data issues

###### 2.2 Fix any issues discovered
    Fix data types errors with:

    Single column    --> .withColumn()     col().cast() or use this expr('cast()')

    Multiple columns --> .withColumns({ }) col().cast() or use this expr('cast()')

    example: 
       # Fix the order_date as it was incorrectly inferred
       df_fixed2 =  df2_raw.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")  ) 

       # FIX MORE THAN ONE COLUMN at the same time
       #-------------------OPTION ONE --Using col()----------------
       #df_fixed2 =  df2_raw.withColumns({"order_date": to_date(col("order_date"), "yyyy-MM-dd"),
       #                                  "order_id": col("order_id").cast("string")
                                         #"order_id": col("order_id").try_cast(IntegerType())  try_cast returns null if an error hapens  
       #                                 })

       #-------------------OPTION TWO --Using expr()----------------
       #df_fixed2 =  df2_raw.withColumns({"order_date": to_date(col("order_date"), "yyyy-MM-dd"),
       #                                  "order_id": expr("cast(order_id as string)")
       #                                 })



#### 3 Query the data the SQL Query way and the PySpark Transformation way
  -   SQL Query way
  -   PySpark transformations way






#### 4 Example

--------------------------------------

##### Step One: Creating spark session

 spark.version

##### Step Two: Creating datframe
First of all you must create a dataframe and here we list 3 different ways, although we will describe 2 in this notebook, the 3rd one is discussed in a previous nortebook.

 1 create a Dataframe by reading a file in the Volume(a csv, json, etc.)

-     using spark.read.format
                      .option
                      .option
                      .load

2 Create a dataframe by reading a table where the data sits

-     using spark.read.table

3 Check the notebook "3_Customer_Dataframe_Creation". We are not reviewing this one for this practice

-     using spark.createDataFrame(data=data_list, schema=data_schema)


##### Step Three: Query the data SQL Query way and PySpark Transformations way
------------------------------------------------------------------------------------------
###### SQL Query way --> Write a normal SQL query 

###### PySpark transformations way --> IDEAL to perform the steps of an SQL query

 -    < 1 Read teh data

 -    < 2 Apply transformations(querying the data)
        - IDEAL :    ENCAPSULATES ALL TRANSFORMATIONS INTO ONE dataframe
        - NOT IDEAL: Creates a dataframe per transformation

 -    < 3 Show/Execute the result/Actions(applied to teh result)

####5 Practices

![image_1774032783996.png](./image_1774032783996.png "image_1774032783996.png")
#### 1 Create spark session


##### Creating spark session

In [0]:
# We will use this to create a spark session from here onwards
spark.version

'4.1.0'

--------------------------------------------------------
##### installing some libraries

In [0]:
pip install duckdb pandas # install duckdb and pandas to be able to query a dataframe

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


-----------------------------------------
#####Restarting python 

In [0]:
dbutils.library.restartPython() # tHIS restarts the kernel or python after running the above command to install duckdb

-----------------------------------------
#####Importing some libraries

In [0]:
import pandas as pd
import duckdb
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import to_date, col, expr, try_to_date, regexp_replace

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
### 2 create dataframe 

##### (by reading a file from Volume (csv, json, etc.) )


In [0]:
# Reading a csv file
#file_df = ( spark.read.format('csv')
#                      .option('header', 'true')
#                      .option('inferSchema', 'true')
#                      .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/sf-fire-calls.csv")
#          )

#Read a json file
#Using a connector(options).
#json_file_df = (
#                spark.read.format('json')
#                .load(path= '/Volumes/dev/spark_db/datasets/spark_programming/data/diamonds.json')
#              )


-----------------------------------------------------------------------------
##### <> using spark.read.format (FOOL)


 <> create a Dataframe by reading a file (in the Volume a csv, json, etc.) 
-     using spark.read.format
                      .option
                      .option
                      .load         

In [0]:

raw_emp_df = ( spark.read.format('csv')
                          .option("header", True)
                          .option("inferSchema", True)
                          .load("/Volumes/dev/spark_db/datasets/spark_programming/data/employee.csv")
              )

raw_emp_df.display()

employee_id,Name,departmentid,salary,startdate,enddate
1,John Smith,1,60000,2020-01-15,null
2,Sarah Johnson,1,65000,2019-06-20,null
3,Michael Brown,2,75000,2018-03-10,null
4,Emily White,2,70000,2021-02-14,null
5,David Lee,3,80000,2017-11-25,null
6,Jennifer Davis,3,78000,2019-09-01,2023-03-30
7,Robert Wilson,null,55000,2022-04-12,null
8,Lisa Anderson,4,72000,2020-07-08,null
9,James Taylor,4,71000,2021-01-20,null
10,Mary Martinez,null,58000,2022-05-15,null


--------------------------------------------------------
##### 2.1 Eyeball the data to spot possible issues

------------------------------------------------------------------
#####2.2 Fix any data issues discovered.
After reading the data we noticed that one issue needs to be solved.


 In case you need to make fixes you can make use of these : 

 From PySpark use either withColumn() or withColumns()

1 withColumn() to add a column or replacing the existing column that has the same name. 

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html

2 withColumns() to add multiple columns or replacing the existing columns that have the same names.

doc: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html



In [0]:
fix_emp_df= raw_emp_df.withColumns({ "departmentid": col("departmentid").try_cast(IntegerType())                                       
                                     #,"enddate": to_date(col("enddate"), "yyyy-MM-dd")
                                   }
                                  )
fix_emp_df.display()

employee_id,Name,departmentid,salary,startdate,enddate
1,John Smith,1,60000,2020-01-15,null
2,Sarah Johnson,1,65000,2019-06-20,null
3,Michael Brown,2,75000,2018-03-10,null
4,Emily White,2,70000,2021-02-14,null
5,David Lee,3,80000,2017-11-25,null
6,Jennifer Davis,3,78000,2019-09-01,2023-03-30
7,Robert Wilson,null,55000,2022-04-12,null
8,Lisa Anderson,4,72000,2020-07-08,null
9,James Taylor,4,71000,2021-01-20,null
10,Mary Martinez,null,58000,2022-05-15,null


##### DEPARETMENT DF

In [0]:
raw_dep_df = ( spark.read.format('csv')
                          .option("header", True)
                          .option("inferSchema", True)
                          .load("/Volumes/dev/spark_db/datasets/spark_programming/data/department.csv")
              )

raw_dep_df.display()

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
### 3 Query  the data the SQL Query way and the PySpark Transformation way

##### Before anything you MUST create a dataframe



![image_1774032783996.png](./image_1774032783996.png "image_1774032783996.png")

#### 10-50 SQL Interview Questions 

https://www.linkedin.com/feed/update/urn:li:activity:7402591561454383104/?updateEntityUrn=urn%3Ali%3Afs_updateV2%3A%28urn%3Ali%3Aactivity%3A7402591561454383104%2CFEED_DETAIL%2CEMPTY%2CDEFAULT%2Cfalse%29



![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")

#### Second highest salary by employee
------------------------------------------------------------------
##### SQL Query

In [0]:
%sql
SELECT MAX(SALARY) as SECOND_TOP_SALARY
FROM   dev.spark_db.employee as 
WHERE SALARY < (SELECT MAX(SALARY) FROM dev.spark_db.employee)

------------------------------------------------------------------
##### PySpark

![image_1774556836253.png](./image_1774556836253.png "image_1774556836253.png")



###### 1 Read teh data.

In [0]:
# 1 This steps is done when we created the dataframe by reading a file from Volume "raw_cust_df"

###### 2 Apply transformations(Composable query: All steps in encapsualted) 

##### Now you have to use the newly created dataframe "raw_cust_df"


#####Remember: If using select() and selectExpr() 
  - select() uses the DataFrame API's column objects and functions
  - selectExpr() accepts SQL-style expressions as strings.
    -     But be careful when using groupBy() and agg(). See the notebook "Instructions..."


###STANDARD WAY
##### The piece of code below is the STANDARD WAY of performing groupBy and aggregations 

#### Option One

In [0]:

from pyspark.sql.functions import to_date, col, expr, sum, count, max 

max_salary = raw_emp_df.agg(max("salary")).first()[0] # max_salary

# Then filter for salaries less than max and get the second highest
result_df = (raw_emp_df.filter(col("salary") < max_salary)
                       .agg(max("salary").alias("SECOND_TOP_SALARY"))
            )

result_df.display()

# First, get the maximum salary
"""Esta línea de código se utiliza comúnmente en
PySpark (o Spark con Python) para extraer un único valor numérico —en este caso, el salario más alto— de un DataFrame y almacenarlo en una variable de Python.
Desglose del código:

    raw_emp_df: Es el nombre del DataFrame que contiene los datos de los empleados.
    .agg(max("salary")):
        agg es la función de agregación.
        max_("salary") calcula el valor máximo de la columna llamada "salary".
        Nota: El resultado de este paso sigue siendo un DataFrame con una sola fila y una sola columna.
    .first():
        Esta es una "acción" de Spark que toma la primera fila del DataFrame resultante y la devuelve como un objeto tipo Row de Spark
"""

##error:
##### This piece of code below will produce an error cause 

you cannot replicate the query directly using a selectExper() as it requires the registration of the dataframe as a temporary view first


Use the standard way instead

In [0]:
"""
from pyspark.sql.functions import to_date, col, expr, sum, count

result_df =(  raw_cust_df.selectExpr("productid" ,"sum(quantity * price) as Revenue")
                         .orderBy("Revenue", ascending=False)
                         .limit(3)
           )
            
result_df.display()
"""       

#### Option Two

Creating a temporary view

In [0]:
from pyspark.sql.functions import expr, col, count

# Assuming 'raw_cust_df.createOrReplaceTempView("customers")' is your PySpark DataFrame
raw_emp_df.createOrReplaceTempView("employee")

# Run the SQL query using spark.sql()
sql_query = """
SELECT MAX(SALARY) as SECOND_TOP_SALARY
FROM employee
WHERE SALARY < (SELECT MAX(SALARY) FROM employee)
"""
result_df_sql = spark.sql(sql_query)

# Show the result
result_df_sql.show()

In [0]:
"""from pyspark.sql.functions import to_date, col, expr, sum, count


result_df =(  raw_cust_df.groupBy("productid")
                         .agg(expr("sum(quantity * price)  as Revenue"))
                         .orderBy("Revenue", ascending=False)
                         .limit(3)
           )
            
result_df.display()
"""


In [0]:
"""
from pyspark.sql.functions import expr

result_df = ( cust_df.select("customerid", "customer_name", "productid", "quantity", "price")
                     .where("productid IS NOT NULL")
                     .groupBy("productid").agg(expr("sum(quantity * price)").alias("sales"))
                     .orderBy("sales", ascending=False)
                     .limit(3)
            )

result_df.display()            
"""

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### Find employees without department

-------------------------------------------------------------
##### SQL Query

In [0]:
%sql
SELECT DISTINCT EMPLOYEE_ID
FROM dev.spark_db.employee as e
WHERE DEPARTMENTID IS NULL

--------------------------------------------------------
##### Pyspark

##### Option One
###### Standard way

In [0]:

from pyspark.sql.functions import col, expr, regexp_replace

result_emp_df =(  raw_emp_df.where(col("departmentid").isNull())
                            .select(col("employee_id"))
               ) 
result_emp_df.display()

employee_id
7
10
13
16
19


##### Option Two
###### Creating a temporaryview

In [0]:
from pyspark.sql.functions import col, expr

#Create a temporaryView
raw_emp_df.createOrReplaceTempView("employee")

sql_query= """
          SELECT EMPLOYEE_ID
          FROM dev.spark_db.employee as e
          WHERE DEPARTMENTID IS NULL
           """

result_emp_df= spark.sql(sql_query)
result_emp_df.show()

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
### TBD

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### NUMBER TBD

-------------------------------------------------------------
##### SQL Query

--------------------------------------------------------
##### Pyspark

######Method 1: Using DataFrame API (Recommended)
This approach is the most common and "PySpark way" to achieve the result.

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### 1 TBD

-------------------------------------------------------------
##### SQL Query

--------------------------------------------------------
##### Pyspark

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### 1 TBD

-------------------------------------------------------------
##### SQL Query

--------------------------------------------------------
##### Pyspark

In [0]:
from pyspark.sql.functions import col, count


# 1. Group by 'customerid' and count the occurrences, giving the count column an alias
#    (e.g., 'customer_count')
grouped_df = (raw_cust_df.groupBy("customerid") \
                        .agg(count("*").alias("customer_count")) #
             )
# 2. Filter the result to keep only rows where the count is greater than 1
result_df = grouped_df.filter(col("customer_count") > 1) #

# 3. Show the result
result_df.show()


######Method 2: Using Spark SQL

You can also use raw SQL directly within PySpark by registering your DataFrame as a temporary view.

In [0]:
from pyspark.sql.functions import expr, col, count

# Assuming 'raw_cust_df.createOrReplaceTempView("customers")' is your PySpark DataFrame
raw_cust_df.createOrReplaceTempView("customers")

# Run the SQL query using spark.sql()
sql_query = """
SELECT customerid, count(*) as customer_count
FROM customers
GROUP BY customerid
HAVING count(*) > 1
"""
result_df_sql = spark.sql(sql_query)

# Show the result
result_df_sql.show()


### Practica #1: Number of distint products types

#### With SQL query

In [0]:
%sql
SELECT COUNT(DISTINCT PRODUCTID) AS DISTINCT_PRODUCT_COUNT 
FROM dev.spark_db.customers


### With Pyspark transformation

##### Take the same piece of the SQL query above : 

COUNT(DISTINCT PRODUCTID) AS DISTINCT_PRODUCT_COUNT 

##### and, in pyspark, put into the:

 selectExpr('COUNT(DISTINCT PRODUCTID) AS DISTINCT_PRODUCT_COUNT ')

In [0]:
res_cust_df = (cust_df.selectExpr('count(distinct productid) as distinct_product_count') 
              )
res_cust_df.display()

### Practice#2: Customers with consecutive purchases (2 days)

#### SQL query

In [0]:
%sql
WITH Prev_pur as ( SELECT CUSTOMERID, ORDER_ID, ORDER_DATE, LAG(ORDER_DATE) OVER( ORDER BY ORDER_DATE ) AS Prev_purchase 
                   FROM dev.spark_db.customers
                   ORDER BY CUSTOMERID
                 )
             SELECT *
             FROM Prev_pur
             WHERE (PREV_PURCHASE + INTERVAL 1 DAY) = ORDER_DATE
             --DATEDIFF(ORDER_DATE , Prev_purchase ) =1 
;

#### Pyspark

In [0]:
query = """
        WITH Prev_pur as ( SELECT CUSTOMERID, ORDER_ID, ORDER_DATE, LAG(ORDER_DATE) OVER( ORDER BY ORDER_DATE ) AS Prev_purchase 
                   FROM dev.spark_db.customers
                   ORDER BY CUSTOMERID
                 )
             SELECT *
             FROM Prev_pur
             WHERE (PREV_PURCHASE + INTERVAL 1 DAY) = ORDER_DATE
             --DATEDIFF(ORDER_DATE , Prev_purchase ) =1 
        """
result= spark.sql(query)
display(result)

In [0]:

data_schema = "id int, source string , destination string, distance int"

data_list= [(101, "Mumbai", "Goa", 587),
            (102, "Mumbai", "Bangalore", 985),
            (102, "Mumbai", "Bangalore", 985),
            (103, "Dheli", "Chennai", 2208),
            (104, "Dheli", "Chennai", 2208),     
            (105, "Bangalore", "Kolkata", 1868),             
            (105, "Bangalore", "Kolkata", 1865)                           
            ]
#df = spark.createDataFrame(data=data_list, schema=data_schema)
df=pd.DataFrame(data_list)

df.display()

In [0]:
query = """
SELECT id, source
from df 
"""

result = duckdb.query(query).df()
print(result)

In [0]:

%sql



